<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week2/DLP_Week2_GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# !pip install -U "datasets==3.6.0"

In [3]:
from datasets import load_dataset
from pprint import pprint

Download the BookCorpus dataset. Take every 7-th sample (the indices are multiple of `7:[0,7,14,21,...])` from the entire dataset. This will result in a dataset with 10 million samples (exactly, 10,572,033). Use these samples to build a tokenizer with the BPE tokenization algorithm by varying the vocabulary size.

- Normalizer: LowerCase
- PreTokenizer: WhiteSpace
- Model: BPE
- Special tokens: [GO],[UNK],[PAD],[EOS]
- PostProcessing: None

Tokenize the input text: “SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.” using the following configurations

In [ ]:
ds = load_dataset('bookcorpus/bookcorpus', split='all')
pprint(ds)

README.md:   0%|          | 0.00/18.5k [00:00<?, ?B/s]

bookcorpus.py:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

The repository for bookcorpus/bookcorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bookcorpus/bookcorpus.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

In [ ]:
ds = ds.select(range(0, len(ds), 7))
pprint(len(ds))

10572033


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

In [ ]:
text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

## Question 1: Keep the vocabulary size at 5000 and tokenize the input text using the learned vocabulary. Choose the number of tokens returned by the tokenizer.

In [ ]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.normalizer = normalizers.Lowercase()
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=5000,
    special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"]
)

In [ ]:
def batch_iterator(batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i + batch_size]["text"]

tokenizer.train_from_iterator(
    batch_iterator(),
    trainer=trainer
)

In [ ]:
output = tokenizer.encode(text)

print("Tokens:", output.tokens)
print("Token IDs:", output.ids)
print("Token count:", len(output.ids))

Tokens: ['se', 'bi', 'study', 'find', 's', '9', '3', '%', 'of', 'indi', 'vid', 'ual', 'f', '&', 'o', 'tr', 'ad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '2', '2', 'and', 'fy', '2', '4', '.']
Token IDs: [108, 1386, 3463, 567, 65, 33, 27, 13, 100, 4669, 1580, 1071, 52, 14, 61, 398, 130, 246, 440, 129, 3750, 716, 4560, 26, 26, 91, 4560, 26, 28, 22]
Token count: 30


## Question 3

In [ ]:
from tokenizers.models import BPE

In [ ]:
trained = Tokenizer(BPE()).from_file('/content/hopper.json')
tokens = trained.encode(text).tokens
print(tokens)
print("Number of tokens: ", len(tokens) )

['se', 'bi', 'study', 'find', 's', '9', '3', '%', 'of', 'indi', 'vid', 'ual', 'f', '&', 'o', 'tr', 'ad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '2', '2', 'and', 'fy', '2', '4', '.']
Number of tokens:  30


In [ ]:
from tokenizers import Tokenizer

trained = Tokenizer.from_file("hopper.json")

text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

tokens = trained.encode(text).tokens

print(set(tokens))
print("Number of tokens:", len(set(tokens)))

{'find', 'tr', 'of', '9', 'and', 'made', 'ers', 'se', '&', 'lo', 'study', '.', 'f', 'ual', 'fy', '%', 'sses', '4', '3', '2', 's', 'bi', 'between', 'ad', 'indi', 'o', 'vid'}
Number of tokens: 27


## Question 4

Suppose we know that the acronym “FY” will likely appear very frequently in most of the input text (assume the text comes from the financial domain). Therefore, we hope that adding it manually to the vocabulary might help. Add the token “FY” to the vocabulary and tokenize (use the Hopper tokenizer) the input text. Enter the number of tokens produced.

In [ ]:
from tokenizers import Tokenizer

# Load the pretrained Hopper tokenizer
trained = Tokenizer.from_file("hopper.json")

# Add FY to the vocabulary
trained.add_tokens(["FY"])

# Input text
text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

# Tokenize the input
tokens = trained.encode(text).tokens

print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

Tokens: ['se', 'bi', 'study', 'find', 's', '9', '3', '%', 'of', 'indi', 'vid', 'ual', 'f', '&', 'o', 'tr', 'ad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '2', '2', 'and', 'fy', '2', '4', '.']
Number of tokens: 30
